# DeerStealer DLL Sideloading Through Signed Binaries

Hunting for DeerStealer/DeerLoader delivery chains across customer environments using LCQL queries in LimaCharlie, with results triaged in DataWrangler.

**Date:** 2026-03-19  
**Author:** Josh Strickland  
**Platform:** LimaCharlie EDR  

## Hypothesis

Unsigned binaries and DLLs appearing in `ProgramData` subfolders that don't belong to known, installed software indicate DLL sideloading activity.

**Basis:** [Red Canary documented a DeerStealer variant](https://redcanary.com/blog/threat-intelligence/phishing-rmm-tools/) using renamed `DicomPortable.exe` (Apowersoft-signed) sideloading a malicious `Qt5Core.dll`, delivered via MSI to a non-standard `ProgramData` directory. The sideloaded DLL is always unsigned even though the binary loading it is legitimately signed. That's the differentiator.

## Objectives

1. Query all customer organizations for unsigned binaries and DLLs in `ProgramData` using `CODE_IDENTITY` events over a 14-day window
2. Triage results in DataWrangler by extracting and deduplicating on the first `ProgramData` subfolder
3. Identify any subfolders that don't map to known, installed software
4. Pivot into LimaCharlie timeline for any outliers to determine if the activity is malicious

## Method

Query `CODE_IDENTITY` events across all customer organizations using LCQL via the LimaCharlie SDK. `CODE_IDENTITY` fires once per unique hash+path combination, keeping volume manageable. Filter to Windows sensors, `ProgramData` file paths, and unsigned binaries (`FILE_IS_SIGNED != 1`). Run queries in parallel across orgs using `ThreadPoolExecutor`, collect into a Pandas DataFrame, expand the nested JSON event data into flat columns, and export to CSV for triage in VS Code DataWrangler.

**Triage approach:** Extract the first subfolder under `ProgramData` from each file path, filter out known customer-specific software, deduplicate on folder name. Any remaining folder that doesn't map to a recognized vendor is an outlier worth investigating.

---

### Setup

Install dependencies and load credentials. The LimaCharlie SDK is pinned to v4.x because v5 removed the `Manager`/`Replay` classes used for LCQL queries.

In [ ]:
# Install requirements into the running kernel (no restart needed)
# Pin limacharlie to v4.x — v5 removed Manager/Replay classes
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "limacharlie", "-y"], stderr=subprocess.DEVNULL)
subprocess.check_call([sys.executable, "-m", "pip", "install", "limacharlie<5", "pandas", "python-dotenv", "--quiet", "--no-cache-dir"])
print("Requirements installed")

In [ ]:
import limacharlie
from limacharlie.Replay import Replay
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import json
import time
import os
from pathlib import Path
from datetime import datetime, timezone
from dotenv import load_dotenv

# Load .env from notebook directory
load_dotenv()
print("Imports loaded, .env processed")

### Credentials

Reads `LC_UID` and `LC_API_KEY` from environment variables (loaded from `.env` file automatically).

Create a `.env` file in the same directory as this notebook:
```
LC_UID=your-user-id-here
LC_API_KEY=your-api-key-here
```

In [ ]:
LC_UID = os.environ.get("LC_UID", "")
LC_API_KEY = os.environ.get("LC_API_KEY", "")

assert LC_UID, "LC_UID not found — set it in your .env file or system environment variables"
assert LC_API_KEY, "LC_API_KEY not found — set it in your .env file or system environment variables"
print(f"UID: {LC_UID[:8]}...")
print("Credentials loaded successfully")

### Discover Organizations

Auto-discover every org the API key has access to via `Manager.userAccessibleOrgs()`. No per-org configuration needed.

In [ ]:
# Create a user-level Manager (no specific org needed)
mgr = limacharlie.Manager(
    uid=LC_UID,
    secret_api_key=LC_API_KEY
)

# Fetch all orgs this user has access to (default limit is 10, set high to get all)
result = mgr.userAccessibleOrgs(with_names=True, limit=500)
orgs = dict(zip(result['orgs'], [result['names'].get(oid, 'Unknown') for oid in result['orgs']]))

print(f"Found {len(orgs)} organizations:")
for oid, name in sorted(orgs.items(), key=lambda x: x[1]):
    print(f"  {name}: {oid[:12]}...")

### Filter orgs (optional)

Uncomment and modify to exclude or target specific orgs.

In [ ]:
# --- Uncomment below to exclude specific orgs ---
# EXCLUDE_ORGS = {"example1", "example2"}
# orgs = {oid: name for oid, name in orgs.items() if name not in EXCLUDE_ORGS}

# --- Uncomment below to target only specific orgs ---
# INCLUDE_ONLY = {"customer-a", "customer-b"}
# orgs = {oid: name for oid, name in orgs.items() if name in INCLUDE_ONLY}

print(f"Querying {len(orgs)} org(s)")

### LCQL Query

The hunt query: unsigned `CODE_IDENTITY` events in `ProgramData` across all Windows sensors over 14 days (`-336h`).

LCQL syntax reference:
- **Timeframe:** `-30m`, `-1h`, `-24h`, `-336h`, or absolute `2026-03-18 00:00:00 to 2026-03-19 00:00:00` (UTC)
- **Sensor selectors:** `*` (all), `plat == windows`, `plat == linux`, `plat == macos`
- **Filter operators:** `contains`, `not contains`, `is`, `is not`, `starts with`, `ends with`, `matches` (regex)

In [ ]:
# =====================================================
# EDIT YOUR QUERY HERE
# =====================================================

LCQL_QUERY = "-336h | plat == windows | CODE_IDENTITY | event/FILE_PATH contains 'ProgramData' and event/SIGNATURE/FILE_IS_SIGNED != 1"

# Max events per org (to prevent runaway queries)
LIMIT_PER_ORG = 5000

# Max events LC will scan per org (lower = faster but may miss results)
LIMIT_EVENT_SCAN = 50000

# Max concurrent org queries
MAX_WORKERS = 15

print(f"Query: {LCQL_QUERY}")
print(f"Limit per org: {LIMIT_PER_ORG}")

### Run Query Across All Orgs

Queries each org in parallel using `ThreadPoolExecutor`. Each org gets its own `Manager` and `Replay` instance. Results are flattened into rows with routing metadata (org, hostname, sensor ID, timestamp) and the raw event JSON. The `ext-cloud-cli` internal sensor is filtered out.

In [ ]:
def query_single_org(oid, org_name):
    """Run LCQL against one org, return list of flat event dicts."""
    try:
        mgr = limacharlie.Manager(
            oid=oid,
            uid=LC_UID,
            secret_api_key=LC_API_KEY
        )
        replay = Replay(mgr)

        response = replay._doQuery(
            LCQL_QUERY,
            limitEvent=LIMIT_EVENT_SCAN,
            limitEval=None,
            isCursorBased=False,
            stream='event'
        )

        error = response.get('error', '')
        results = response.get('results', [])

        if error and not results and 'limit' not in error.lower():
            return [], f"Error: {error}"

        events = []
        for result in results:
            evt = result.get('data', result) if isinstance(result, dict) else None
            if not evt:
                continue

            routing = evt.get('routing', {})
            hostname = routing.get('hostname', '')

            # Skip internal CLI sensor
            if hostname == 'ext-cloud-cli':
                continue

            # Build flat row
            ts = routing.get('event_time', 0)
            if ts > 9999999999:
                ts = ts / 1000

            row = {
                'org_name': org_name,
                'oid': oid,
                'timestamp': datetime.fromtimestamp(ts, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S') if ts else '',
                'event_type': routing.get('event_type', ''),
                'hostname': hostname,
                'sensor_id': routing.get('sid', ''),
                'event_data': json.dumps(evt.get('event', {}), default=str),
            }
            events.append(row)

        # Sort newest first, apply limit
        events.sort(key=lambda e: e['timestamp'], reverse=True)
        return events[:LIMIT_PER_ORG], None

    except Exception as e:
        return [], str(e)


# Run in parallel
all_events = []
errors = []
start = time.time()

workers = min(MAX_WORKERS, len(orgs))
print(f"Querying {len(orgs)} orgs with {workers} workers...")

with ThreadPoolExecutor(max_workers=workers) as executor:
    futures = {
        executor.submit(query_single_org, oid, name): (oid, name)
        for oid, name in orgs.items()
    }

    completed = 0
    for future in as_completed(futures):
        oid, name = futures[future]
        events, error = future.result()
        completed += 1

        if error:
            errors.append({'org': name, 'oid': oid, 'error': error})
            print(f"  [{completed}/{len(orgs)}] {name}: ERROR - {error[:80]}")
        else:
            all_events.extend(events)
            print(f"  [{completed}/{len(orgs)}] {name}: {len(events)} events")

elapsed = time.time() - start
print(f"\nDone in {elapsed:.1f}s — {len(all_events)} total events, {len(errors)} errors")

### Build DataFrame

Collect all events into a Pandas DataFrame sorted by timestamp (newest first). This is the base dataset before event JSON expansion.

In [ ]:
df = pd.DataFrame(all_events)

if not df.empty:
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    df = df.sort_values('timestamp', ascending=False).reset_index(drop=True)

print(f"{len(df)} rows, {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")
df.head(10)

### Expand Event Data

Flatten the nested JSON `event_data` column into individual columns (prefixed with `evt_`). This makes fields like `evt_FILE_PATH`, `evt_SIGNATURE.FILE_IS_SIGNED`, etc. available for filtering and analysis in DataWrangler.

In [ ]:
# Expand the JSON event_data column into separate columns
if not df.empty and 'event_data' in df.columns:
    event_expanded = pd.json_normalize(df['event_data'].apply(json.loads))
    # Prefix expanded columns to avoid collisions
    event_expanded.columns = ['evt_' + c for c in event_expanded.columns]
    df_expanded = pd.concat([df.drop(columns=['event_data']), event_expanded], axis=1)
    print(f"Expanded: {len(df_expanded.columns)} columns")
    df_expanded.head(5)
else:
    df_expanded = df
    print("No data to expand")

### Quick Analysis

Event distribution by org and event type. Helps identify which orgs are generating the most noise before triage.

In [ ]:
if not df.empty:
    print("Events per org:")
    print(df['org_name'].value_counts().to_string())
    print(f"\nEvent types:")
    print(df['event_type'].value_counts().head(20).to_string())
    print(f"\nTime range: {df['timestamp'].min()} to {df['timestamp'].max()}")
else:
    print("No results")

### Export to CSV

Export the expanded DataFrame to CSV for triage in VS Code DataWrangler. The expanded columns make it possible to filter, sort, and deduplicate on individual event fields without parsing JSON.

In [ ]:
# Base DataFrame (event_data as JSON string — good for DataWrangler)
# Export the expanded DataFrame with proper columns
OUTPUT_FILE = f"lcql_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df_expanded.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE} ({len(df_expanded)} rows, {len(df_expanded.columns)} columns)")

# Expanded DataFrame (flattened event columns — good for pandas analysis)
# OUTPUT_EXPANDED = f"lcql_results_expanded_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
# df_expanded.to_csv(OUTPUT_EXPANDED, index=False)
# print(f"Saved: {OUTPUT_EXPANDED} ({len(df_expanded)} rows)")

### Errors

Any orgs that failed to query (auth issues, timeouts, etc.).

In [ ]:
if errors:
    print(f"{len(errors)} org(s) had errors:")
    for e in errors:
        print(f"  {e['org']}: {e['error']}")
else:
    print("No errors")

---

## Findings

**Result: True Positive**

2,821 events over 14 days. After deduplicating on the first `ProgramData` subfolder in DataWrangler, all folders mapped to known vendors except one: `AppDownload`.

`C:\ProgramData\AppDownload\` contained a full DeerStealer delivery chain:

| File | Description |
|------|-------------|
| `EngineX_Co64.exe` | Legitimate Comodo-signed binary  |
| `cmdres.dll` | Malicious sideloaded DLL, unsigned  |
| `Bairrout.xd` | Encrypted HijackLoader shellcode |
| `Kleanmean.py` | HijackLoader configuration |

**Kill chain:** MSI installer (`em_73g2oeim_installer.msi`) dropped `EngineX_Co64.exe` to a temp GUID folder, which self-relocated to `ProgramData\AppDownload\` and sideloaded `cmdres.dll`. The DLL decrypted shellcode from `Bairrout.xd` and loaded HijackLoader, which started `SecureLoad_test.exe` (legitimate Q-Dir binary by Nenad Hrg) in a suspended state and injected DeerStealer into it. DeerStealer never touched disk as its own executable.

**C2 confirmation:** `SecureLoad_test.exe` made a DNS request to `sciecdn.cfd`, a domain correlated to DeerStealer C2 infrastructure on VirusTotal. The DNS request PPID matched the SecureLoad_test.exe PID.

### MITRE ATT&CK

- **T1218.007** - System Binary Proxy Execution: Msiexec
- **T1574.002** - Hijack Execution Flow: DLL Side-Loading
- **T1036.005** - Masquerading: Match Legitimate Name or Location
- **T1555.003** - Credentials from Password Stores: Credentials from Web Browsers
- **T1539** - Steal Web Session Cookie

## References

- [Red Canary - Phishing for RMM Tools](https://redcanary.com/blog/threat-intelligence/phishing-rmm-tools/) - DicomPortable.exe / Qt5Core.dll sideloading variant
- [eSentire - DeerStealer Analysis](https://www.esentire.com/blog/dont-get-caught-in-the-headlights-deerstealer-analysis) - Full decryption chain, module stomping, and injection mechanics
- [eSentire DeerStealer IOCs on GitHub](https://github.com/eSentire/iocs/blob/main/DeerStealer/DeerStealer-IoCs-06-03-2025.txt)
- [HijackLibs - Qt5Core.dll](https://hijacklibs.net/entries/3rd_party/qt/qt5core.html) - Known DLL sideloading target
- [HijackLibs Project](https://hijacklibs.net/) - Community-maintained DLL sideloading target database
- [Any.Run Sandbox Analysis](https://app.any.run/tasks/8cbd1b88-aa31-4de3-b46c-c4ae778a3238) - Full behavioral breakdown of the MSI installer